In [11]:
!pip install google-api-python-client pandas tqdm youtube-transcript-api

In [12]:
import time
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

In [ ]:
# =========================
# 1. 配置
# =========================
API_KEY = "YOUR KEY"   # 这里替换成你的 key

SEARCH_KEYWORDS = [
    "phone case review",
    "laptop sleeve review",
    "bag organizer review"
]

MAX_VIDEOS_PER_KEYWORD = 10
MAX_COMMENTS_PER_VIDEO = 100
LANGUAGE_PREFERENCE = ["en"]   # transcript优先语言，可改成 ["en", "zh-Hans", "zh-Hant"]

# 输出文件名
VIDEO_OUTPUT = "youtube_videos.csv"
COMMENT_OUTPUT = "youtube_comments.csv"
TRANSCRIPT_OUTPUT = "youtube_transcripts.csv"

In [14]:
# =========================
# 2. 初始化 YouTube API client
# =========================
youtube = build("youtube", "v3", developerKey=API_KEY)


In [15]:
# =========================
# 3. 搜索视频
# =========================
def search_videos(keyword, max_results=10, order="relevance"):
    """
    根据关键词搜索视频，返回 video_id 列表
    order可选: relevance, date, rating, viewCount
    """
    video_ids = []
    next_page_token = None

    while len(video_ids) < max_results:
        remaining = min(50, max_results - len(video_ids))

        request = youtube.search().list(
            part="snippet",
            q=keyword,
            type="video",
            maxResults=remaining,
            order=order,
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response.get("items", []):
            video_id = item["id"]["videoId"]
            video_ids.append(video_id)

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    return video_ids

In [16]:
# =========================
# 4. 批量获取视频详情
# =========================
def get_video_details(video_ids):
    """
    批量获取视频详细信息
    返回 list[dict]
    """
    all_video_data = []

    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]

        request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(batch_ids)
        )
        response = request.execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            content = item.get("contentDetails", {})

            video_data = {
                "video_id": item.get("id"),
                "title": snippet.get("title"),
                "description": snippet.get("description"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "tags": ", ".join(snippet.get("tags", [])) if snippet.get("tags") else None,
                "category_id": snippet.get("categoryId"),
                "default_audio_language": snippet.get("defaultAudioLanguage"),
                "duration": content.get("duration"),
                "view_count": int(stats.get("viewCount", 0)) if stats.get("viewCount") else 0,
                "like_count": int(stats.get("likeCount", 0)) if stats.get("likeCount") else 0,
                "comment_count": int(stats.get("commentCount", 0)) if stats.get("commentCount") else 0,
                "url": f"https://www.youtube.com/watch?v={item.get('id')}"
            }
            all_video_data.append(video_data)

        time.sleep(0.2)

    return all_video_data



In [17]:
# =========================
# 5. 获取评论（含 replies）
# =========================
def get_comments(video_id, max_comments=100):
    """
    获取某个视频的 top-level comments 和 replies
    返回 list[dict]
    """
    comments_data = []
    next_page_token = None
    fetched = 0

    while fetched < max_comments:
        remaining = min(100, max_comments - fetched)

        request = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=min(100, remaining),
            pageToken=next_page_token,
            textFormat="plainText",
            order="relevance"
        )

        try:
            response = request.execute()
        except Exception as e:
            print(f"[Comment Error] video_id={video_id}, error={e}")
            break

        items = response.get("items", [])
        if not items:
            break

        for item in items:
            top_comment = item["snippet"]["topLevelComment"]
            top_snippet = top_comment["snippet"]

            comments_data.append({
                "video_id": video_id,
                "comment_type": "top_level",
                "comment_id": top_comment.get("id"),
                "parent_comment_id": None,
                "author": top_snippet.get("authorDisplayName"),
                "author_channel_url": top_snippet.get("authorChannelUrl"),
                "text": top_snippet.get("textDisplay"),
                "like_count": top_snippet.get("likeCount", 0),
                "published_at": top_snippet.get("publishedAt"),
                "updated_at": top_snippet.get("updatedAt")
            })
            fetched += 1

            if fetched >= max_comments:
                break

            # replies
            replies = item.get("replies", {}).get("comments", [])
            for reply in replies:
                reply_snippet = reply["snippet"]

                comments_data.append({
                    "video_id": video_id,
                    "comment_type": "reply",
                    "comment_id": reply.get("id"),
                    "parent_comment_id": reply_snippet.get("parentId"),
                    "author": reply_snippet.get("authorDisplayName"),
                    "author_channel_url": reply_snippet.get("authorChannelUrl"),
                    "text": reply_snippet.get("textDisplay"),
                    "like_count": reply_snippet.get("likeCount", 0),
                    "published_at": reply_snippet.get("publishedAt"),
                    "updated_at": reply_snippet.get("updatedAt")
                })

        next_page_token = response.get("nextPageToken")
        if not next_page_token or fetched >= max_comments:
            break

        time.sleep(0.2)

    return comments_data



In [18]:
# =========================
# 6. 获取 transcript
# =========================
def get_transcript(video_id, language_preference=None):
    """
    获取 transcript，返回:
    - transcript_text
    - transcript_language
    - transcript_available
    """
    if language_preference is None:
        language_preference = ["en"]

    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

        transcript = None

        # 优先找指定语言
        for lang in language_preference:
            try:
                transcript = transcript_list.find_transcript([lang])
                break
            except:
                pass

        # 如果没找到指定语言，尝试自动生成/任意 transcript
        if transcript is None:
            try:
                transcript = next(iter(transcript_list))
            except:
                return {
                    "video_id": video_id,
                    "transcript_available": False,
                    "transcript_language": None,
                    "transcript_text": None
                }

        transcript_data = transcript.fetch()
        transcript_text = " ".join([seg["text"] for seg in transcript_data])

        return {
            "video_id": video_id,
            "transcript_available": True,
            "transcript_language": getattr(transcript, "language_code", None),
            "transcript_text": transcript_text
        }

    except (TranscriptsDisabled, NoTranscriptFound):
        return {
            "video_id": video_id,
            "transcript_available": False,
            "transcript_language": None,
            "transcript_text": None
        }
    except Exception as e:
        print(f"[Transcript Error] video_id={video_id}, error={e}")
        return {
            "video_id": video_id,
            "transcript_available": False,
            "transcript_language": None,
            "transcript_text": None
        }

In [19]:
# =========================
# 7. 主流程
# =========================
def run_pipeline(
    search_keywords,
    max_videos_per_keyword=10,
    max_comments_per_video=100,
    language_preference=None
):
    all_video_ids = []
    keyword_video_map = []

    print("Step 1: Searching videos...")
    for keyword in tqdm(search_keywords):
        video_ids = search_videos(keyword, max_results=max_videos_per_keyword)
        for vid in video_ids:
            keyword_video_map.append({
                "search_keyword": keyword,
                "video_id": vid
            })
        all_video_ids.extend(video_ids)

    # 去重
    all_video_ids = list(dict.fromkeys(all_video_ids))
    keyword_map_df = pd.DataFrame(keyword_video_map)

    print(f"Total unique videos found: {len(all_video_ids)}")

    print("Step 2: Fetching video details...")
    videos_data = get_video_details(all_video_ids)
    videos_df = pd.DataFrame(videos_data)

    # 合并 keyword 信息
    keyword_agg = (
        keyword_map_df.groupby("video_id")["search_keyword"]
        .apply(lambda x: " | ".join(sorted(set(x))))
        .reset_index()
    )
    videos_df = videos_df.merge(keyword_agg, on="video_id", how="left")

    print("Step 3: Fetching comments...")
    all_comments = []
    for vid in tqdm(all_video_ids):
        comments = get_comments(vid, max_comments=max_comments_per_video)
        all_comments.extend(comments)
    comments_df = pd.DataFrame(all_comments)

    print("Step 4: Fetching transcripts...")
    all_transcripts = []
    for vid in tqdm(all_video_ids):
        transcript_result = get_transcript(vid, language_preference=language_preference)
        all_transcripts.append(transcript_result)
    transcripts_df = pd.DataFrame(all_transcripts)

    print("Step 5: Saving outputs...")
    videos_df.to_csv(VIDEO_OUTPUT, index=False, encoding="utf-8-sig")
    comments_df.to_csv(COMMENT_OUTPUT, index=False, encoding="utf-8-sig")
    transcripts_df.to_csv(TRANSCRIPT_OUTPUT, index=False, encoding="utf-8-sig")

    print("Done.")
    print(f"Saved: {VIDEO_OUTPUT}, {COMMENT_OUTPUT}, {TRANSCRIPT_OUTPUT}")

    return videos_df, comments_df, transcripts_df


In [20]:
# =========================
# 8. 执行
# =========================
videos_df, comments_df, transcripts_df = run_pipeline(
    search_keywords=SEARCH_KEYWORDS,
    max_videos_per_keyword=MAX_VIDEOS_PER_KEYWORD,
    max_comments_per_video=MAX_COMMENTS_PER_VIDEO,
    language_preference=LANGUAGE_PREFERENCE
)

print("\n=== videos_df sample ===")
display(videos_df.head())

print("\n=== comments_df sample ===")
display(comments_df.head())

print("\n=== transcripts_df sample ===")
display(transcripts_df.head())

Step 1: Searching videos...


100%|██████████| 3/3 [00:01<00:00,  1.56it/s]


Total unique videos found: 30
Step 2: Fetching video details...
Step 3: Fetching comments...


100%|██████████| 30/30 [00:16<00:00,  1.87it/s]


Step 4: Fetching transcripts...


...

,video_id,title,description,channel_id,channel_title,published_at,tags,category_id,default_audio_language,duration,view_count,like_count,comment_count,url,search_keyword
0,QQunUM9BMH4,I’ve Tested 100s of Cases… Here Are the 4 iPho...,I’ve tested 100’s of iPhone cases BUT because ...,UC1hiI8TEoA0db5HhU4gbsUg,MobileReviewsEh,2025-09-21T19:15:07Z,"mobilereviewseh, iphone 17 cases, best iphone ...",28,en,PT7M37S,549143,8334,774,https://www.youtube.com/watch?v=QQunUM9BMH4,phone case review
1,Bb4lj6K7HkY,Spigen Ultra Hybrid MagFit iPhone 17 Review – ...,"When it comes to budget iPhone 17 cases, I’ve ...",UC1hiI8TEoA0db5HhU4gbsUg,MobileReviewsEh,2025-11-01T15:30:07Z,"30-drop test, MagSafe protection, Spigen MagFi...",28,en,PT4M3S,103354,1465,187,https://www.youtube.com/watch?v=Bb4lj6K7HkY,phone case review
2,Z4VNAbshz1Y,$2 vs $200 iPhone Case 📲,,UCLVihavmaUONy0SeOX_F7aw,SarahGrace,2024-04-26T15:30:10Z,None,22,en,PT1M,13983957,601039,3369,https://www.youtube.com/watch?v=Z4VNAbshz1Y,phone case review
3,VXegNKdO95U,"Bright case, bright mood ☀️ #phonecase #iphone...",,UCYhsa_sKlw9aAr6Co_oWEbw,Burga,2025-08-18T16:00:47Z,None,26,en,PT19S,96328498,693409,5885,https://www.youtube.com/watch?v=VXegNKdO95U,phone case review
4,JA0Dj4Mlhuo,I Was Wrong About Budget iPhone 17 Cases – The...,I tested 50+ iPhone 17 cases. I know what the ...,UC1hiI8TEoA0db5HhU4gbsUg,MobileReviewsEh,2026-02-09T16:30:16Z,"mobilereviewseh, iPhone 17 cases, best budget ...",28,en,PT8M23S,59641,1646,170,https://www.youtube.com/watch?v=JA0Dj4Mlhuo,phone case review



=== comments_df sample ===


,video_id,comment_type,comment_id,parent_comment_id,author,author_channel_url,text,like_count,published_at,updated_at
0,QQunUM9BMH4,top_level,Ugw_UPKg8NsqacjKjNt4AaABAg,None,@muhammadalialtaf7842,http://www.youtube.com/@muhammadalialtaf7842,THANK YOU FOR DOING REAL REVIEWS. So many peop...,72,2025-09-23T00:33:59Z,2025-09-23T00:33:59Z
1,QQunUM9BMH4,top_level,Ugx72vND1NYzW7MESTt4AaABAg,None,@user-zg4ir8ug3s,http://www.youtube.com/@user-zg4ir8ug3s,You should design your own case!,55,2025-11-08T12:14:14Z,2025-11-08T12:14:14Z
2,QQunUM9BMH4,reply,Ugx72vND1NYzW7MESTt4AaABAg.APGOXuw9gXDAR3iDXHtMdJ,Ugx72vND1NYzW7MESTt4AaABAg,@PunzL,http://www.youtube.com/@PunzL,yeah like anyone can just design a case if the...,0,2025-12-23T07:07:36Z,2025-12-23T07:07:36Z
3,QQunUM9BMH4,reply,Ugx72vND1NYzW7MESTt4AaABAg.APGOXuw9gXDARArwVSwk3k,Ugx72vND1NYzW7MESTt4AaABAg,@hhy_au,http://www.youtube.com/@hhy_au,@PunzLhe can,3,2025-12-26T01:47:13Z,2025-12-26T01:47:13Z
4,QQunUM9BMH4,reply,Ugx72vND1NYzW7MESTt4AaABAg.APGOXuw9gXDARC_3cSsd_r,Ugx72vND1NYzW7MESTt4AaABAg,@Fidexus,http://www.youtube.com/@Fidexus,"@PunzLdude, he can",2,2025-12-26T17:40:47Z,2025-12-26T17:40:47Z



=== transcripts_df sample ===


,video_id,transcript_available,transcript_language,transcript_text
0,QQunUM9BMH4,False,None,None
1,Bb4lj6K7HkY,False,None,None
2,Z4VNAbshz1Y,False,None,None
3,VXegNKdO95U,False,None,None
4,JA0Dj4Mlhuo,False,None,None
